# 01. Opacus Fundamentals - DP-SGD and Hyperparameters

In Notebook 00, we successfully wrapped our PyTorch objects. But as we saw, the privacy budget ($\epsilon$) remains undefined until we actually train the model. 

In this notebook, we will:
1. Build a complete DP-SGD training loop.
2. Observe how the privacy accountant tracks $\epsilon$ over time.
3. Understand the key hyperparameters (`noise_multiplier` and `max_grad_norm`).

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from opacus import PrivacyEngine

# 1. Setup Data, Model, and Optimizer
# setup data
X = torch.randn(500, 10)
y = torch.randint(0, 2, (500,))
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32)

# setup model
model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 2)
)

# setup optimizer & loss
optimizer = optim.SGD(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

In [2]:
# 2. Initializing Privacy Engine
privacy_engine = PrivacyEngine()

model_dp, optimizer_dp, dataloader_dp = privacy_engine.make_private(
    module = model,
    optimizer = optimizer,
    data_loader = dataloader,
    noise_multiplier = 1.2, # Adds noise to gradients
    max_grad_norm = 1.0, # Clips gradients
)

C:\Users\MUBASHIR MOHSIN\AppData\Roaming\Python\Python39\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


## The **"DP-SGD"** Training Loop

The loop looks almost identical to standard PyTorch loop. The actual magic happens inside optimizer_dp.step().

Every time the `step()` is called, Opacus:
1. Computes the $L_2$ norm of the gradients for each sample in the batch.
2. Clips any gradient that exceeds `max_grad_norm`.
3. Aggregates the clipped gradients.
4. Adds Gaussian noise proportional to `noise_multiplier`.
5. Updates the model weights and logs the step in the step in the privacy accountant.

In [3]:
EPOCHS = 10
DELTA = 1e-3

for epoch in range(EPOCHS):
    model_dp.train()
    total_loss = 0.0
    
    for data, target in dataloader_dp:
        optimizer_dp.zero_grad()
        output = model_dp(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer_dp.step() # privacy accountant updates
        
        total_loss += loss.item()
        
    # check the budget at the end of each epoch
    epsilon = privacy_engine.get_epsilon(delta=DELTA)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(dataloader_dp):.4f} | Spent ε: {epsilon:.4f}")

Epoch 1/10 | Loss: 0.7128 | Spent ε: 0.8072
Epoch 2/10 | Loss: 0.7060 | Spent ε: 1.1184
Epoch 3/10 | Loss: 0.6948 | Spent ε: 1.3664
Epoch 4/10 | Loss: 0.6983 | Spent ε: 1.5816
Epoch 5/10 | Loss: 0.6881 | Spent ε: 1.7760
Epoch 6/10 | Loss: 0.6945 | Spent ε: 1.9553
Epoch 7/10 | Loss: 0.6856 | Spent ε: 2.1233
Epoch 8/10 | Loss: 0.6896 | Spent ε: 2.2821
Epoch 9/10 | Loss: 0.7015 | Spent ε: 2.4336
Epoch 10/10 | Loss: 0.7019 | Spent ε: 2.5787


### Observations
* <b> Epsilon ($\epsilon$) strictly increases. </b> Every time the model sees data, privacy leaks slightly.
* **The Utility-Privacy Tradeoff:** If you increase the `noise_multiplier` _(more privacy, lower $\epsilon$)_, the loss will likely be higher or more unstable _(less accuracy)_.

## Practical Implications in Research

In dynamic budget allocation, you won't always use a fixed `noise_multiplier`. You will eventually need to monitor $\epsilon$ dynamically and adjust training parameters on the fly before a strict budget is exhausted. The baseline training loop here is your starting point for those advanced adjustments.

In the next notebook, we will look at Model Compatability. Not all standard PyTorch layers work with **DP-SGD** out of the box (like standard BatchNorm). We will learn how Opacus's `ModuleValidator` handles this.

### 🧪 Experiment 01: The Utility-Privacy Tradeoff on Real Data
Open the `Experiment_01.ipynb` file alongside this tutorial for a complete, end-to-end implementation of DP-SGD using a real-world patient tabular dataset. In this experiment, we load the raw CSV, engineer the features, and define a binary classification model to predict a patient's target condition. We train two identical models side-by-side: a standard baseline model and a Differentially Private model utilizing the Opacus PrivacyEngine. Finally, we compare the loss and accuracy metrics to actively observe the utility loss (the drop in predictive performance) incurred when enforcing a strict privacy budget (𝜖). This serves as our practical foundation for understanding model behavior under DP constraints.